In [1]:
import requests
import time
import warnings
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from typing import List, Optional, Tuple, Dict, Any, Union
from tqdm.notebook import tqdm
from io import StringIO

# Загрузка данных

In [2]:
# Получу список тикеров S&P500
response = requests.get("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies", 
                        headers={
                                 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 \
                                 (KHTML, like Gecko) Chrome/116.0 Safari/537.36"
                        })
response.raise_for_status()
tickers = pd.read_html(StringIO(response.text))[0]['Symbol'].tolist()
tickers = [t.replace('.', '-') for t in tickers]

def get_data(tickers, 
             retries=3):
    if not tickers or retries <= 0:
        return pd.DataFrame()

    data = yf.download(tickers, 
                       period="max", 
                       auto_adjust=False, 
                       progress=True,
                       multi_level_index=True, 
                       threads=True)

    # Рекурсивно догружаю то, что не загрузилось
    broken = list(data.isna().mean()[lambda x: x == 1.].to_frame().reset_index().iloc[:, -2].unique())
    if broken:
        data2 = get_data(broken, retries=retries-1)
        data = pd.concat([data, data2], axis=1)

    data.drop(columns=data.isna().mean().sort_values()[lambda x: x == 1.].index, inplace=True)

    return data

In [3]:
data = get_data(tickers)

[*********************100%***********************]  503 of 503 completed


In [4]:
df = data.copy()

# Анализ пропусков

In [5]:
df.xs('A', axis=1, level=1)

Price,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
1962-01-02,NaN,NaN,NaN,NaN,NaN,NaN
1962-01-03,NaN,NaN,NaN,NaN,NaN,NaN
1962-01-04,NaN,NaN,NaN,NaN,NaN,NaN
1962-01-05,NaN,NaN,NaN,NaN,NaN,NaN
1962-01-08,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
2026-02-23,124.199997,124.199997,124.260002,121.680000,121.830002,2359400.0
2026-02-24,124.400002,124.400002,125.830002,123.129997,124.510002,2729600.0
2026-02-25,124.720001,124.720001,125.589996,123.440002,125.510002,2469300.0


In [6]:
# Получим индексы с NaN
for i in range(df.shape[1]):
    df1 = df.iloc[:, i]
    res = df1.loc[df1.first_valid_index():].isna().mean().mean()
    if res != 0:
        indexes = df1.loc[df1.first_valid_index():][df1.loc[df1.first_valid_index():].isna()].index
        name = df1.loc[df1.first_valid_index():][df1.loc[df1.first_valid_index():].isna()].name
        print(name, indexes)

('Adj Close', 'FISV') DatetimeIndex(['2025-11-12'], dtype='datetime64[ns]', name='Date', freq=None)
('Adj Close', 'HUBB') DatetimeIndex(['1972-06-06', '1972-06-07', '1972-06-08', '1972-06-09',
               '1972-06-12', '1972-06-14', '1972-06-15', '1972-06-19',
               '1972-06-20', '1972-06-22',
               ...
               '1973-02-02', '1973-02-05', '1973-02-06', '1973-02-09',
               '1973-02-13', '1973-02-14', '1973-02-15', '1973-02-16',
               '1973-02-20', '1977-08-08'],
              dtype='datetime64[ns]', name='Date', length=121, freq=None)
('Adj Close', 'LMT') DatetimeIndex(['1974-06-03'], dtype='datetime64[ns]', name='Date', freq=None)
('Close', 'FISV') DatetimeIndex(['2025-11-12'], dtype='datetime64[ns]', name='Date', freq=None)
('Close', 'HUBB') DatetimeIndex(['1972-06-06', '1972-06-07', '1972-06-08', '1972-06-09',
               '1972-06-12', '1972-06-14', '1972-06-15', '1972-06-19',
               '1972-06-20', '1972-06-22',
               .

1) FISV — нет данных на 2025‑11‑12

На ноябрь 2025 произошло корпоративное событие и смена листинга/тикера у Fiserv: компания меняла листинг и тикер, переходя с NYSE на Nasdaq, с новым тикером FISV.

2) По другим тикерам данные слишком старые (70е годы - данные могли не записаться)

Поэтому можно эти пропуски заполнить методом ffill

In [7]:
df = df.ffill()

# Корреляционный анализ

In [8]:
log_returns = df['Adj Close'].pct_change()
corrs = log_returns.corr().stack()

In [9]:
# Оставлю уникальные пары

corrs = corrs[corrs.index.get_level_values(0) != corrs.index.get_level_values(1)]
corrs.index = pd.MultiIndex.from_tuples([tuple(sorted(idx)) for idx in corrs.index])
corrs = corrs[~corrs.index.duplicated()]

In [10]:
corrs.sort_values(ascending=False).head(20)

GOOG  GOOGL    0.997337
FOX   FOXA     0.987517
NWS   NWSA     0.970780
CFG   KEY      0.895950
      FITB     0.893847
      RF       0.891364
DOW   LYB      0.876397
CFG   HBAN     0.868867
      PNC      0.866562
      TFC      0.864690
NCLH  RCL      0.846513
HLT   MAR      0.846490
CCL   NCLH     0.843800
AVB   EQR      0.843358
COF   SYF      0.836711
CFG   USB      0.834084
MA    V        0.832411
DAL   UAL      0.828893
MPC   VLO      0.828813
MET   PRU      0.822987
dtype: float64

### Очень высокая корреляция — **разные классы одной компании**

1. **GOOG – GOOGL (0.997)** → два класса акций Google (C и A) → почти идентичные движения.
2. **FOX – FOXA (0.988)** → два класса акций Fox Corp (A и B).
3. **NWS – NWSA (0.971)** → два класса акций News Corp (A и B).

---

### Высокая корреляция — **один сектор / похожий бизнес**

4. **CFG – KEY (0.896)** → банки (Citigroup и KeyCorp) → чувствуют ставки и финансовый сектор одинаково.
5. **CFG – FITB (0.894)** → банки (Citigroup и Fifth Third Bank).
6. **CFG – RF (0.891)** → банки (Citigroup и Regions Financial).
7. **DOW – LYB (0.876)** → химическая отрасль (Dow и LyondellBasell) → реагируют на сырьё и макроэкономику.
8. **CFG – HBAN (0.869)** → банки (Citigroup и Huntington Bancshares).
9. **CFG – PNC (0.867)** → банки (Citigroup и PNC).
10. **CFG – TFC (0.865)** → банки (Citigroup и Truist).
11. **NCLH – RCL (0.847)** → круизные линии (Norwegian Cruise и Royal Caribbean) → чувствительны к туризму и сезонности.
12. **HLT – MAR (0.846)** → гостиничный сектор (Hilton и Marriott).
13. **CCL – NCLH (0.844)** → круизные линии (Carnival и Norwegian Cruise).
14. **AVB – EQR (0.843)** → недвижимость/REIT (AvalonBay и Equity Residential).
15. **COF – SYF (0.837)** → финансовые услуги/кредитные карты (Capital One и Synchrony).
16. **CFG – USB (0.834)** → банки (Citigroup и U.S. Bancorp).
17. **MA – V (0.832)** → платёжные системы (Mastercard и Visa) → реагируют на транзакции потребителей и экономическую активность.
18. **DAL – UAL (0.829)** → авиаперевозки (Delta и United Airlines) → чувствительны к топливу, туризму, экономике.
19. **MPC – VLO (0.829)** → нефтепереработка (Marathon Petroleum и Valero) → реагируют на цены на нефть.
20. **MET – PRU (0.823)** → страховые компании (MetLife и Prudential) → финансы, акции страхового сектора движутся схоже.

---

💡 **Вывод по списку:**

* **0.97–1.0** → разные классы одной компании
* **0.85–0.95** → компании одного сектора / близкий бизнес (банки, круизы, гостиницы, нефть, платежи, REIT)



In [11]:
corrs.sort_values(ascending=True).head(20)

PG    Q     -0.313648
ACGL  Q     -0.276755
GEV   GIS   -0.274003
AWK   GEV   -0.248482
GIS   Q     -0.245825
CB    Q     -0.237652
BRO   Q     -0.229811
CPB   GEV   -0.225637
ED    Q     -0.219293
      GEV   -0.215821
ERIE  Q     -0.212696
AFL   Q     -0.206919
Q     T     -0.204643
AIG   Q     -0.196289
CAG   GEV   -0.196007
CBOE  GEV   -0.194643
MRSH  Q     -0.190668
AON   Q     -0.188043
KO    Q     -0.183906
AJG   Q     -0.183493
dtype: float64

### Топ отрицательной корреляции

1. **PG – Q (-0.314)** → Procter & Gamble (потребительские товары) и QuantumScape (технологии, батареи) → разные сектора → противоположные движения.
2. **ACGL – Q (-0.277)** → страхование (Arch Capital) vs технологии → разные реакции на макро.
3. **GEV – GIS (-0.274)** → Gran Tierra Energy (энергетика) vs General Mills (потребительские товары) → энергия более волатильная, еда стабильная.
4. **AWK – GEV (-0.248)** → коммунальные услуги vs энергия → противоположная динамика.
5. **GIS – Q (-0.246)** → потребительские товары vs технологии → разные тренды.
6. **CB – Q (-0.238)** → финансы (Chubb) vs технологии → разные драйверы.
7. **BRO – Q (-0.230)** → страхование vs технологии.
8. **CPB – GEV (-0.226)** → Campbell Soup (еда) vs энергия → циклы противоположные.
9. **ED – Q (-0.219)** → коммунальные услуги vs технологии → defensive vs growth.
10. **ED – GEV (-0.216)** → коммунальные услуги vs энергия → стабильно vs волатильно.
11. **ERIE – Q (-0.213)** → страхование vs технологии.
12. **AFL – Q (-0.207)** → страхование vs технологии.
13. **Q – T (-0.205)** → технологии vs телеком → иногда разные рыночные реакции.
14. **AIG – Q (-0.196)** → страхование vs технологии.
15. **CAG – GEV (-0.196)** → потребительские товары vs энергия.
16. **CBOE – GEV (-0.195)** → финансовый сектор vs энергия.
17. **MRSH – Q (-0.191)** → страхование (Marsh & McLennan) vs технологии.
18. **AON – Q (-0.188)** → страхование vs технологии.
19. **KO – Q (-0.184)** → Coca-Cola (потребительские товары) vs технологии.
20. **AJG – Q (-0.183)** → страхование vs технологии.

---

### Вывод

* Все пары **из разных секторов** → отрицательная корреляция.
* Q - новый актив - данные за последние пару лет против данных за десятиления, локально низкая корреляция

## По Секторам

In [12]:
general_info = pd.read_html(StringIO(response.text))

In [13]:
general_info = general_info[0]

In [14]:
general_info['GICS Sector'].unique()

array(['Industrials', 'Health Care', 'Information Technology',
       'Utilities', 'Financials', 'Materials', 'Consumer Discretionary',
       'Real Estate', 'Communication Services', 'Consumer Staples',
       'Energy'], dtype=object)

Перевод:
* **Industrials** — Промышленность
* **Health Care** — Здравоохранение
* **Information Technology** — Информационные технологии
* **Utilities** — Коммунальные услуги (энергетика, водоснабжение, газ)
* **Financials** — Финансовый сектор
* **Materials** — Сырьё и материалы
* **Consumer Discretionary** — Потребительские товары и услуги необязательного спроса
* **Real Estate** — Недвижимость
* **Communication Services** — Коммуникационные услуги
* **Consumer Staples** — Потребительские товары первой необходимости
* **Energy** — Энергетика

In [65]:
log_returns = df['Adj Close'].pct_change()
sector_map = general_info.set_index('Symbol')['GICS Sector']

log_returns.columns = pd.MultiIndex.from_arrays(
    [
        log_returns.columns.map(sector_map),  # верхний уровень — Sector
        log_returns.columns                  # нижний уровень — Ticker
    ],
    names=['Sector', 'Ticker']
)

sectors_returns = log_returns.copy().groupby(level='Sector', axis=1).mean()
corrs = log_returns.groupby(level='Sector', axis=1).mean().corr().stack()
corrs = corrs[corrs.index.get_level_values(0) != corrs.index.get_level_values(1)]
corrs.index = pd.MultiIndex.from_tuples([tuple(sorted(idx)) for idx in corrs.index])
corrs = corrs[~corrs.index.duplicated()]

corrs.sort_values()

/var/folders/tr/d703srhj765104bxxcbz75lr0000gn/T/ipykernel_2020/3352493889.py:12: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  sectors_returns = log_returns.copy().groupby(level='Sector', axis=1).mean()
/var/folders/tr/d703srhj765104bxxcbz75lr0000gn/T/ipykernel_2020/3352493889.py:13: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  corrs = log_returns.groupby(level='Sector', axis=1).mean().corr().stack()


Consumer Discretionary  Utilities                 0.339038
                        Energy                    0.390539
Communication Services  Utilities                 0.394816
Information Technology  Utilities                 0.395176
Communication Services  Energy                    0.425468
                        Consumer Discretionary    0.469102
Energy                  Utilities                 0.475704
Consumer Discretionary  Consumer Staples          0.479268
Energy                  Information Technology    0.484009
Consumer Staples        Energy                    0.489100
Energy                  Real Estate               0.496020
Materials               Utilities                 0.500927
Consumer Discretionary  Health Care               0.501890
Health Care             Utilities                 0.502002
Consumer Discretionary  Materials                 0.503373
                        Information Technology    0.506090
Communication Services  Materials                 0.5065

### 1. Consumer Discretionary

* Наименее коррелирует с **Utilities (0.34)** — лучшая диверсификация.
* Средняя корреляция с другими секторами ~0.50–0.60.
* Высокая корреляция с **Financials (0.59)** и **Industrials (0.58)** — циклический характер.

---

### 2. Communication Services

* Средние корреляции с **Utilities (0.39)** и Energy (0.43).
* Выше среднего с **Consumer Staples (0.52)** и **Financials (0.62)**.
* Высокая корреляция с **Industrials (0.60)** и IT (0.58).

---

### 3. Information Technology

* Средняя корреляция с **Utilities (0.40)**.
* Более высокая с **Materials (0.59)**, Real Estate (0.54), Financials (0.66) и Health Care (0.66).
* Циклический сектор с сильной связью с экономикой и рынком.

---

### 4. Energy

* Средняя корреляция с **Utilities (0.48)**.
* Высокая с **Industrials (0.62)**, Financials (0.59), Materials (0.61), Real Estate (0.50).
* Сектор чувствителен к циклу сырья и макроэкономике.

---

### 5. Health Care

* Средняя корреляция с Utilities (0.50).
* Высокая с **Industrials (0.72)**, IT (0.66), Financials (0.70).
* Отчасти циклический, но более устойчивый к кризисам.

---

### 6. Consumer Staples

* Средняя корреляция с Utilities (0.58).
* Более высокая с **Materials (0.62)**, Financials (0.66), Health Care (0.67).
* Более защитный сектор, но частично синхронизирован с рынком.

---

### 7. Industrials

* Высокая корреляция со всеми циклическими секторами: Financials (0.82), Materials (0.75), IT (0.72), Real Estate (0.69).
* Менее диверсифицирует портфель, чем защитные сектора.

---

### 8. Financials

* Очень высокая корреляция с **Industrials (0.82)**, Materials (0.75), Health Care (0.71).
* Циклический сектор, сильная зависимость от экономического роста и ставок.

---

### 9. Materials

* Средняя корреляция с Utilities (0.50).
* Высокая с Industrials (0.75), Financials (0.75), Real Estate (0.64).
* Циклический сектор, чувствителен к спросу на сырьё и инвестициям.

---

### 10. Utilities

* Наименьшая корреляция с Consumer Discretionary (0.33).
* Средняя с другими секторами ~0.40–0.55.
* Защитный сектор, устойчивая доходность, снижает волатильность портфеля.

---

### 11. Real Estate

* Средние корреляции с Utilities (0.54) и Materials (0.64).
* Высокие с Financials (0.71) и Industrials (0.69).
* Циклический сектор, чувствителен к ставкам и экономике.

---

### 12. Energy

* В основном высокие корреляции с Materials, Industrials и Financials (0.60–0.62).
* Сильная зависимость от циклов сырья и глобальной экономики.


## По Суб секторам

In [66]:
log_returns = df['Adj Close'].pct_change()
sector_map = general_info.set_index('Symbol')['GICS Sub-Industry']

log_returns.columns = pd.MultiIndex.from_arrays(
    [
        log_returns.columns.map(sector_map),  # верхний уровень — Sector
        log_returns.columns                  # нижний уровень — Ticker
    ],
    names=['Sub-Sector', 'Ticker']
)

sub_sectors_returns = log_returns.copy().groupby(level='Sub-Sector', axis=1).mean()
corrs = log_returns.groupby(level='Sub-Sector', axis=1).mean().corr().stack()
corrs = corrs[corrs.index.get_level_values(0) != corrs.index.get_level_values(1)]
corrs.index = pd.MultiIndex.from_tuples([tuple(sorted(idx)) for idx in corrs.index])
corrs = corrs[~corrs.index.duplicated()]

corrs.sort_values()

/var/folders/tr/d703srhj765104bxxcbz75lr0000gn/T/ipykernel_2020/458046497.py:12: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  sub_sectors_returns = log_returns.copy().groupby(level='Sub-Sector', axis=1).mean()
/var/folders/tr/d703srhj765104bxxcbz75lr0000gn/T/ipykernel_2020/458046497.py:13: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  corrs = log_returns.groupby(level='Sub-Sector', axis=1).mean().corr().stack()


Heavy Electrical Equipment      Water Utilities                         -0.248482
                                Packaged Foods & Meats                  -0.163181
                                Telecom Tower REITs                     -0.131987
                                Integrated Telecommunication Services   -0.131110
                                Household Products                      -0.128448
                                                                           ...   
Electric Utilities              Multi-Utilities                          0.784641
Industrial REITs                Multi-Family Residential REITs           0.805917
Diversified Banks               Regional Banks                           0.836863
Office REITs                    Retail REITs                             0.848862
Multi-Family Residential REITs  Office REITs                             0.853875
Length: 7875, dtype: float64

### Отрицательные корреляции (диверсификация)

* **Heavy Electrical Equipment — Water Utilities**: -0.25
* **Heavy Electrical Equipment — Packaged Foods & Meats**: -0.16
* **Heavy Electrical Equipment — Telecom Tower REITs / Integrated Telecommunication Services / Household Products**: -0.13 … -0.13

**Вывод:** Heavy Electrical Equipment плохо коррелирует с защитными и потребительскими сегментами, что обеспечивает естественную диверсификацию.

---

### Средние корреляции (0.40–0.70)

* Обычно встречаются между разными субсекторами внутри одного крупного сектора, например:

  * Utilities ↔ различные типы REITs
  * Consumer Staples ↔ Health Care
* Эти связи отражают умеренную синхронизацию доходностей по экономическим и рыночным факторам.

---

### Высокие корреляции (0.70–0.85)

* **Electric Utilities — Multi-Utilities**: 0.78
* **Industrial REITs — Multi-Family Residential REITs**: 0.81
* **Diversified Banks — Regional Banks**: 0.83
* **Office REITs — Retail REITs**: 0.85
* **Multi-Family Residential REITs — Office REITs**: 0.85


## Автокорреляции

In [67]:
log_returns = df['Adj Close'].pct_change()

lags = [1, 3, 7, 21, 30, 90, 180, 360]

autocorr_df = pd.DataFrame({
    f'lag_{lag}': log_returns.apply(lambda x: x.autocorr(lag=lag))
    for lag in lags
})

In [68]:
autocorr_df.describe()

,lag_1,lag_3,lag_7,lag_21,lag_30,lag_90,lag_180,lag_360
count,503.000000,503.000000,503.000000,503.000000,503.000000,502.000000,502.000000,501.000000
mean,-0.019292,-0.012348,0.010098,-0.006761,-0.005294,-0.000142,0.004849,-0.000286
std,0.042538,0.023202,0.023218,0.019823,0.021263,0.015807,0.017246,0.015042
min,-0.212821,-0.230394,-0.132881,-0.076890,-0.262708,-0.061762,-0.125725,-0.062587
25%,-0.042627,-0.023832,-0.004356,-0.017474,-0.015401,-0.010184,-0.004183,-0.009276
50%,-0.014426,-0.010564,0.007520,-0.006830,-0.005241,-0.000913,0.005742,-0.001609
75%,0.008680,0.000396,0.021597,0.004158,0.005189,0.008396,0.014339,0.007511
max,0.114271,0.077016,0.114093,0.186384,0.104848,0.118846,0.103993,0.089485


## Среднее значение (mean)

| Лаг     | mean    | Интерпретация                                                                           |
| ------- | ------- | --------------------------------------------------------------------------------------- |
| lag_1   | -0.019  | Небольшое отрицательное значение → слабый возврат к среднему на 1 день.                 |
| lag_3   | -0.012  | Практически 0, память почти отсутствует.                                                |
| lag_7   | 0.010   | Незначительная положительная автокорреляция на неделю → слабые краткосрочные тенденции. |
| lag_21  | -0.006  | Почти нулевая автокорреляция на ~месяц.                                                 |
| lag_30  | -0.005  | Почти нулевая автокорреляция на 1 месяц.                                                |
| lag_90  | -0.000  | На 3 месяца автокорреляция отсутствует.                                                 |
| lag_180 | 0.0048  | Незначительная положительная автокорреляция на полгода.                                 |
| lag_360 | -0.0003 | На год автокорреляция практически 0.                                                    |

**Вывод:** В среднем ряд дневных доходностей акций почти случайный на всех временных горизонтах. Существуют лишь очень слабые краткосрочные тенденции (lag_7).

---

## Стандартное отклонение (std)

| Лаг            | std         | Интерпретация                                                   |
| -------------- | ----------- | --------------------------------------------------------------- |
| lag_1          | 0.0425      | Наибольшая вариативность автокорреляций на один день.           |
| lag_3          | 0.0232      | Разброс снижается → меньшая разница между тикерами.             |
| lag_7          | 0.0232      | Сохраняется умеренный разброс.                                  |
| lag_21…lag_360 | 0.015–0.017 | Разброс между тикерами уменьшается на более длинных горизонтах. |

**Вывод:** На коротких лагах различия между тикерами сильнее. На длинных лагах автокорреляции сходятся к нулю.

---

## Минимальные и максимальные значения (min / max)

* **min**: до -0.26 (lag_30), что указывает на тикеры с заметной отрицательной краткосрочной памятью → возврат к среднему.
* **max**: до 0.18 (lag_21), т.е. некоторые тикеры демонстрируют небольшие тенденции на 1 месяц.

**Вывод:** Есть отдельные «аномальные» тикеры с более выраженной автокорреляцией, но большинство колеблется около нуля.

---

## Квартильная структура (25%, 50%, 75%)

* **50% медиана** почти нулевая на всех лагах → большинство тикеров не имеют автокорреляции.
* **75% квартиль** положительная → несколько тикеров показывают слабую положительную зависимость.
* **25% квартиль** отрицательная → часть тикеров демонстрирует краткосрочный возврат к среднему.

**Вывод:** Распределение автокорреляций симметричное вокруг нуля, слабые положительные/отрицательные отклонения характерны для отдельных тикеров.

---

## Общие выводы

1. **Большинство акций дневной доходности имеют почти нулевую автокорреляцию.**
2. **Слабые положительные автокорреляции** появляются на лаге 7–21 дня у отдельных тикеров → возможные краткосрочные тренды.
3. **Небольшие отрицательные автокорреляции** на лаге 1–3 дня → эффект возврата к среднему у отдельных тикеров.
4. **Разброс (std)** уменьшается с увеличением лага → на длинных горизонтах (3–12 месяцев) автокорреляции практически отсутствуют у всех тикеров.
5. **Вывод для стратегии:** краткосрочные торговые модели могут ориентироваться на отдельные тикеры с выраженной автокорреляцией, но в среднем доходности близки к случайным.



In [71]:
autocorr_df = pd.DataFrame({
    f'lag_{lag}': sectors_returns.apply(lambda x: x.autocorr(lag=lag))
    for lag in lags
})

In [73]:
autocorr_df.describe()

,lag_1,lag_3,lag_7,lag_21,lag_30,lag_90,lag_180,lag_360
count,11.000000,11.000000,11.000000,11.000000,11.000000,11.000000,11.000000,11.000000
mean,0.037943,-0.001238,0.014100,-0.010227,-0.000727,-0.001647,0.008164,-0.001427
std,0.041395,0.009429,0.010342,0.010314,0.012046,0.004694,0.009441,0.005018
min,-0.058494,-0.023281,-0.001572,-0.031619,-0.023526,-0.012598,-0.006443,-0.010840
25%,0.031349,-0.005004,0.006212,-0.012896,-0.005435,-0.003112,0.004803,-0.005147
50%,0.049366,-0.000465,0.012585,-0.009736,0.001903,-0.002289,0.006655,-0.001355
75%,0.060330,0.005479,0.020706,-0.004019,0.005169,0.001073,0.011629,0.001971
max,0.089648,0.009469,0.033490,0.004353,0.019325,0.004346,0.028832,0.005662


## Общая картина

* Диапазон средних автокорреляций по лагам от ~-0.01 до 0.04.
* Как и для отдельных тикеров, **средние доходности по секторам почти случайны на ежедневной основе**.
* Разброс автокорреляций между секторами небольшой (std 0.01–0.04), что указывает на схожее поведение большинства секторов.

---

## Краткосрочные лаги (lag_1, lag_3, lag_7)

| Лаг   | mean    | std   | Интерпретация                                                                                                       |
| ----- | ------- | ----- | ------------------------------------------------------------------------------------------------------------------- |
| lag_1 | 0.0379  | 0.041 | Незначительная положительная память для ряда средних по секторам; вариативность между секторами есть, но небольшая. |
| lag_3 | -0.0012 | 0.009 | Практически нулевая автокорреляция → доходности случайны на 3 дня.                                                  |
| lag_7 | 0.0141  | 0.010 | Слабая положительная автокорреляция на неделю → у некоторых секторов возможны краткосрочные тренды.                 |

**Вывод:** Сектора демонстрируют почти случайное поведение, с небольшими краткосрочными тенденциями у отдельных секторов (lag_1, lag_7).

---

## Среднесрочные и долгосрочные лаги (lag_21–lag_360)

| Лаг     | mean    | std    | Интерпретация                                                             |
| ------- | ------- | ------ | ------------------------------------------------------------------------- |
| lag_21  | -0.010  | 0.010  | Слабая отрицательная автокорреляция → небольшое «возвращение к среднему». |
| lag_30  | -0.0007 | 0.012  | Практически ноль.                                                         |
| lag_90  | -0.0016 | 0.0047 | Нулевая автокорреляция.                                                   |
| lag_180 | 0.0082  | 0.0094 | Незначительная положительная автокорреляция.                              |
| lag_360 | -0.0014 | 0.0050 | Почти ноль.                                                               |

**Вывод:** На горизонте 1–12 месяцев автокорреляции минимальны → долгосрочная память отсутствует, циклы почти не выражены.

# Анализ волатильности

In [78]:
WINDOW = 20  # ~1 торговый месяц
volatility = log_returns.rolling(WINDOW).std()

In [80]:
autocorr_df = pd.DataFrame({
    f'lag_{lag}': volatility.apply(lambda x: x.autocorr(lag=lag))
    for lag in lags
})

In [82]:
autocorr_df.describe()

,lag_1,lag_3,lag_7,lag_21,lag_30,lag_90,lag_180,lag_360
count,503.000000,503.000000,503.000000,503.000000,503.000000,502.000000,502.000000,501.000000
mean,0.983787,0.945731,0.857368,0.526100,0.471252,0.308367,0.250145,0.131082
std,0.008624,0.023077,0.054240,0.158055,0.163820,0.166286,0.140242,0.147984
min,0.872789,0.776593,0.465709,-0.747036,-0.877146,-0.602468,-0.129348,-0.407351
25%,0.980722,0.935467,0.834507,0.452527,0.384354,0.213282,0.157704,0.041614
50%,0.985491,0.950765,0.867333,0.543057,0.482908,0.311077,0.242770,0.109946
75%,0.988430,0.959750,0.890326,0.626454,0.579837,0.415485,0.328795,0.203056
max,0.996579,0.987384,0.961916,0.829668,0.776344,0.701191,0.727045,0.679012


## Общая картина

* Диапазон лагов: от 1 до 360 дней.
* Средние автокорреляции на **малых лагах очень высокие** (~0.98 на lag_1) и постепенно снижаются до ~0.13 на лаге 360.
* **Вывод:** волатильность имеет сильную память, особенно на краткосрочных горизонтах. Это типично для финансовых рынков (волатильность кластеризована).

## Практические выводы

1. **Сильная краткосрочная память волатильности** → подтверждает эффект кластеризации волатильности (волатильные периоды продолжаются несколько дней/недель).
2. **Среднесрочная автокорреляция** → позволяет прогнозировать волатильность на горизонте 1–3 месяца.
3. **Долгосрочные лаги (>6 месяцев)** → автокорреляция снижается, но отдельные экстремальные тикеры сохраняют память.
4. **Применение:**

   * Модели **GARCH / ARCH** для прогнозирования волатильности
   * Оптимизация риска и управление позициями → можно учитывать текущий уровень волатильности, чтобы корректировать экспозицию.
   * Портфельная диверсификация не устраняет кластеризацию волатильности полностью → коррелированные периоды высоких колебаний.


# Асимметрия и эксцесс

## **Kurt (эксцесс, «высота пика»)**

* **Что это:**
  `kurt` измеряет **толщину хвостов и остроту пика** распределения.

* **Интерпретация:**

  * `kurt = 3` → распределение близко к нормальному (mesokurtic)
  * `kurt > 3` → **leptokurtic** (острый пик, тяжёлые хвосты)

    * Часто наблюдается в финансовых рядах → большие экстремальные доходности случаются чаще, чем по нормальному закону
  * `kurt < 3` → **platykurtic** (плоский пик, тонкие хвосты)

**Пример для финансов:**

* Большинство дневных доходностей акций имеют **kurt > 3** → это отражает наличие редких резких скачков, т.е. «черных лебедей».

---

## Применение:

1. **Skew** → сигнал риска в сторону потерь/выгод:

   * Отрицательный skew → риск крупных падений → может быть опасно для портфеля.
   * Положительный skew → потенциально редкие большие выигрыши.

2. **Kurt** → оценка экстремальных событий:

   * Высокий kurt → большие движения более вероятны, чем по нормальному закону.
   * Низкий kurt → редкие экстремальные события.


## По акциям

In [103]:
log_returns.skew().describe()

count    503.000000
mean       0.724911
std        6.834165
min       -4.391534
25%       -0.128127
50%        0.161274
75%        0.468038
max      110.909864
dtype: float64

In [104]:
log_returns.kurt().describe()

count      503.000000
mean        70.162651
std        721.544512
min         -0.421494
25%          8.750117
50%         12.214122
75%         19.235492
max      12702.117143
dtype: float64

## По секторам

In [105]:
sectors_returns.skew()

Sector
Communication Services     0.113480
Consumer Discretionary    15.192454
Consumer Staples          -0.279108
Energy                    -0.266012
Financials                 0.089345
Health Care               -0.239308
Industrials               -0.049564
Information Technology     0.046887
Materials                 -0.186790
Real Estate                0.101610
Utilities                 -0.068510
dtype: float64

In [106]:
sectors_returns.kurt()

Sector
Communication Services       5.436713
Consumer Discretionary    1174.622138
Consumer Staples            11.806037
Energy                      18.072317
Financials                  18.070424
Health Care                  6.951236
Industrials                 14.339518
Information Technology       6.382316
Materials                   10.112543
Real Estate                 23.612432
Utilities                   17.834206
dtype: float64

## По суб секторам

In [107]:
sub_sectors_returns.skew().sort_values()

Sub-Sector
Restaurants                                     -2.730999
Single-Family Residential REITs                 -1.277963
Automotive Parts & Equipment                    -0.796680
Managed Health Care                             -0.586928
Wireless Telecommunication Services             -0.537231
                                                  ...    
Interactive Media & Services                     1.874567
Soft Drinks & Non-alcoholic Beverages            2.236420
Industrial Machinery & Supplies & Components    16.691466
Other Specialty Retail                          33.378986
Homebuilding                                    86.747909
Length: 126, dtype: float64

In [108]:
sub_sectors_returns.kurt().sort_values()

Sub-Sector
Health Care Technology                             2.967925
Specialized Consumer Services                      3.149662
Semiconductor Materials & Equipment                4.128355
Air Freight & Logistics                            4.164478
Technology Hardware, Storage & Peripherals         4.646239
                                                   ...     
Oil & Gas Storage & Transportation                62.297593
Restaurants                                       83.471318
Industrial Machinery & Supplies & Components    1032.609142
Other Specialty Retail                          2149.678472
Homebuilding                                    8685.865638
Length: 126, dtype: float64

## Выводы

### По акциям

**Skew (асимметрия):**

* Среднее: 0.72 → небольшая положительная асимметрия, но **стандартное отклонение 6.83 и max = 110.9** → есть отдельные акции с экстремальной положительной асимметрией (очень редкие резкие всплески доходностей).
* Минимум: -4.39 → отдельные акции имеют сильный отрицательный skew → редкие крупные падения.
* Медиана: 0.16 → большинство акций слегка смещены вправо.

**Kurt (эксцесс):**

* Среднее: 70.16 → крайне высокое → почти все акции имеют тяжёлые хвосты.
* Разброс (std = 721) и max = 12702 → несколько акций с экстремально редкими скачками (черные лебеди).
* 25–75% квартиль: 8.7–19.2 → большинство акций имеют выраженные тяжёлые хвосты.

**Вывод:** доходности акций **не нормальные**, есть редкие экстремальные значения, что сильно влияет на риск и стратегии управления портфелем.

---

### По секторам

**Skew:**

* Большинство секторов близко к нулю: Communication Services (0.11), Financials (0.08), IT (0.046) → распределение почти симметричное.
* Исключение: Consumer Discretionary (15.19) → сильная положительная асимметрия, вероятно, из-за отдельных акций с экстремальным ростом.
* Некоторые сектора слегка отрицательные: Health Care (-0.23), Energy (-0.26) → слабый перекос в сторону потерь.

**Kurt:**

* Большинство секторов имеют умеренно тяжёлые хвосты: Utilities (17.83), Energy (18.07) → экстремальные значения встречаются чаще, чем по нормальному закону.
* Исключение: Consumer Discretionary (1174.6) → крайне тяжёлые хвосты, большой риск экстремальных движений.

**Вывод:** секторная агрегация сглаживает экстремальные индивидуальные акции, но **отдельные секторы (Consumer Discretionary, Real Estate)** всё ещё имеют сильные хвосты и асимметрию.

---

### По субсекторам

**Skew:**

* Негативные экстремумы: Restaurants (-2.73), Single-Family Residential REITs (-1.27), Automotive Parts (-0.79) → высокие риски крупных падений.
* Положительные экстремумы: Homebuilding (86.75), Other Specialty Retail (33.37), Industrial Machinery (16.69) → редкие крупные всплески доходностей.
* **Вывод:** на уровне субсекторов проявляются сильные асимметрии отдельных ниш → высокий риск как падений, так и всплесков.

**Kurt:**

* Минимальные значения: Health Care Technology (2.96), Specialized Consumer Services (3.14) → почти нормальное распределение доходностей.
* Максимальные значения: Homebuilding (8685.86), Other Specialty Retail (2149.68), Industrial Machinery (1032.60) → огромные хвосты → высокая вероятность экстремальных событий.
* **Вывод:** экстремальные события сильно концентрируются на уровне субсекторов и отдельных компаний → важен для риск-менеджмента и стресс-тестов портфеля.

---

## Итоговая интерпретация

1. **Доходности акций**: высокая вариативность, сильно тяжёлые хвосты, отдельные акции с экстремальной асимметрией.
2. **Секторы**: агрегация снижает экстремумы, но отдельные сектора (Consumer Discretionary, Real Estate) сохраняют сильные хвосты и асимметрию.
3. **Субсектора**: на уровне ниш выявляются экстремальные риски (как положительные, так и отрицательные) → важно для детального анализа портфеля.
4. **Вывод для стратегии**: стандартные модели с нормальным распределением доходностей не подходят; нужно учитывать **heavy tails и skew**, особенно для портфелей с концентрацией в определённых секторах или субсекторах.
